# MOHIM motif dataset and stem-wise threshold diagnostics

음원을 source 단위로 한 번 분리해 dataset stem과 앞 30초의 모든 4마디 후보 진단 결과를 함께 저장합니다.

## 0. Drive와 melodysim 브랜치 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'melodysim'
REPO_DIR = Path('/content/MOHIM')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q --no-deps "beat-this @ git+https://github.com/CPJKU/beat_this.git"

## 1. 실험 경로와 범위

In [ ]:
MAX_SONGS = 10
DATA_SOURCE = 'local_dataset'  # 'songs' 또는 'local_dataset'
FORCE_REPROCESS = True
DEVICE = 'cuda'
DEMUCS_BATCH_SIZE = 8
AUDIO_FORMAT = 'flac'
MOTIF_BARS = 4
MOTIF_SEARCH_SECONDS = 30.0
MIN_ACTIVE_RATIO = 0.70
MAX_ONSET_CHROMA_DIFFERENCE = 0.40
MIN_ONSET_SIMILARITY = 0.60
MIN_PITCH_CLASS_SPAN = 0.25
MIN_ONSET_VARIATION = 0.20
MELODYSIM_BATCH_SIZE = 8

SONGS_DIR = Path('/content/drive/MyDrive/MOHIM/songs')
LOCAL_DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/genius_pop_dataset')
MOTIF_DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset')
SONGS_STEM_DIR = Path('/content/drive/MyDrive/MOHIM/songs_stem_dataset')
DIAGNOSTIC_DIR = Path('/content/drive/MyDrive/MOHIM/motif_stem_diagnostics')
BEAT_CHECKPOINT = Path('/content/checkpoints/beat_this_final0.ckpt')
AUDIO_EXTENSIONS = {'.aac', '.flac', '.m4a', '.mp3', '.ogg', '.wav', '.webm'}

assert DATA_SOURCE in {'songs', 'local_dataset'}
MOTIF_DATASET_DIR.mkdir(parents=True, exist_ok=True)
SONGS_STEM_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

## 1-1. `songs` 폴더 입력

`DATA_SOURCE = 'songs'`일 때만 실행되며 파일명을 곡 제목으로 사용합니다.

In [ ]:
if DATA_SOURCE == 'songs':
    song_paths = sorted(
        path for path in SONGS_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
    )[:MAX_SONGS]
    songs = [
        {
            'track_id': path.stem,
            'artist': '',
            'title': path.stem,
            'audio_path': path,
        }
        for path in song_paths
    ]

## 1-2. local dataset 입력

`DATA_SOURCE = 'local_dataset'`일 때 `tracks.json`과 `audio` 폴더에서 실제 음원이 있는 곡을 최대 10개 가져옵니다.

In [ ]:
if DATA_SOURCE == 'local_dataset':
    from mohim.local_dataset import index_audio_files, load_local_tracks, resolve_audio_path

    tracks_json = LOCAL_DATASET_DIR / 'tracks.json'
    audio_dir = LOCAL_DATASET_DIR / 'audio'
    assert tracks_json.is_file(), f'tracks.json이 없습니다: {tracks_json}'
    assert audio_dir.is_dir(), f'음원 폴더가 없습니다: {audio_dir}'

    tracks = load_local_tracks(tracks_json, require_lyrics=True)
    audio_index = index_audio_files(audio_dir)
    songs = []
    for track in tracks:
        audio_path = resolve_audio_path(track, audio_index)
        if audio_path is None:
            print(f'[skip] 음원 없음: {track.artist} - {track.title}')
            continue
        songs.append({
            'track_id': track.track_id,
            'artist': track.artist,
            'title': track.title,
            'audio_path': audio_path,
            'track': track,
        })
        if len(songs) >= MAX_SONGS:
            break
    song_paths = [song['audio_path'] for song in songs]

## 1-3. 선택한 입력 확인

In [ ]:
assert songs, f'사용 가능한 음원이 없습니다: {DATA_SOURCE}'
assert len(songs) == len(song_paths)
print(f'data source: {DATA_SOURCE}')
print('songs:', len(songs))
for song in songs:
    label = f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    print('-', label)

## 2. Demucs와 motif scorer 준비

In [ ]:
import urllib.request
from mohim.motif import MotifConfig, MotifExtractor, create_beat_tracker, onset_variation
from mohim.dataset import DatasetBuilder
from mohim.separator import StemSeparator

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        'https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt', BEAT_CHECKPOINT
    )
separator = StemSeparator(device=DEVICE, model_name='htdemucs_6s')
beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_scorer = MotifExtractor(
    beat_tracker,
    MotifConfig(
        bars=MOTIF_BARS,
        search_seconds=MOTIF_SEARCH_SECONDS,
        min_presence=MIN_ACTIVE_RATIO,
        max_similarity_difference=MAX_ONSET_CHROMA_DIFFERENCE,
        onset_threshold=MIN_ONSET_SIMILARITY,
        pitch_class_span_threshold=MIN_PITCH_CLASS_SPAN,
        onset_variation_threshold=MIN_ONSET_VARIATION,
    ),
)

dataset_builder = (
    DatasetBuilder(
        audio_dir=LOCAL_DATASET_DIR / 'audio',
        output_dir=MOTIF_DATASET_DIR,
        separator=separator,
        motif_extractor=motif_scorer,
        audio_format=AUDIO_FORMAT,
        resume=not FORCE_REPROCESS,
    )
    if DATA_SOURCE == 'local_dataset' else None
)
print('separator and motif scorer ready')

## 3. 곡별 후보 계산·최종 선택·MelodySim 검증

`DEMUCS_BATCH_SIZE`개 곡을 한 GPU batch로 분리합니다. 각 곡은 후보와 최종 motif를 저장하고 최종 후보의 onset variation을 검사한 직후 MelodySim 구간 점수까지 계산·저장합니다. MelodySim 모델은 한 번만 로드해 다음 곡에서도 재사용합니다.

In [ ]:
import json
import librosa
import numpy as np
import pandas as pd
import re
import shutil
from IPython.display import display
from mohim.melodysim import (
    MELODYSIM_CHECKPOINT_FILE,
    MELODYSIM_CHECKPOINT_REPO,
    MELODYSIM_MODEL_ID,
    MelodySimEncoder,
    phase_aligned_four_bar_segments,
)
from mohim.separator import save_audio

def safe_folder_name(value):
    cleaned = re.sub(r'[\\/:*?"<>|\x00-\x1f]', '_', value)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip(' .')
    return cleaned[:120] or 'untitled'

def find_existing_song_dir(audio_path):
    resolved_audio = Path(audio_path).resolve()
    for existing_metadata_path in DIAGNOSTIC_DIR.glob('*/motif_scores.json'):
        try:
            existing_metadata = json.loads(
                existing_metadata_path.read_text(encoding='utf-8')
            )
            if Path(existing_metadata.get('source_audio', '')).resolve() == resolved_audio:
                return existing_metadata_path.parent
        except (OSError, ValueError, TypeError):
            continue
    return None

def song_stem_metadata_path(song):
    display_title = f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    return SONGS_STEM_DIR / safe_folder_name(display_title) / 'metadata.json'

def song_stems_complete(song, audio_path):
    metadata_path = song_stem_metadata_path(song)
    if not metadata_path.is_file():
        return False
    try:
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        same_audio = Path(metadata.get('source_audio', '')).resolve() == Path(audio_path).resolve()
        stem_files = metadata.get('stem_files')
        return (
            same_audio
            and isinstance(stem_files, dict)
            and stem_files
            and all((metadata_path.parent / filename).is_file() for filename in stem_files.values())
        )
    except (OSError, ValueError, TypeError):
        return False

def save_song_stems(song, audio_path, stems, sample_rate):
    metadata_path = song_stem_metadata_path(song)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    stem_files = {}
    for stem_name, stem_audio in stems.items():
        if not stem_name.replace('_', '').isalnum():
            raise ValueError(f'Unsafe separator stem name: {stem_name!r}')
        filename = f'{stem_name}.{AUDIO_FORMAT}'
        save_audio(metadata_path.parent / filename, stem_audio, sample_rate, audio_format=AUDIO_FORMAT)
        stem_files[stem_name] = filename
    metadata = {
        'track_id': song['track_id'], 'artist': song['artist'], 'title': song['title'],
        'source_audio': str(audio_path), 'sample_rate': sample_rate, 'stem_files': stem_files,
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return metadata_path.parent

def internal_candidate_features(audio_path):
    audio, sample_rate = librosa.load(audio_path, sr=None, mono=True)
    hop_length = 512
    chroma = librosa.feature.chroma_cens(
        y=audio, sr=sample_rate, hop_length=hop_length
    )
    chroma_sum = chroma.sum(axis=0, keepdims=True)
    valid_chroma = chroma_sum.ravel() > 1e-8
    chroma_norm = np.divide(
        chroma, chroma_sum, out=np.zeros_like(chroma), where=chroma_sum > 1e-8
    )
    valid_pairs = valid_chroma[:-1] & valid_chroma[1:]
    if np.any(valid_pairs):
        frame_flux = 0.5 * np.sum(np.abs(np.diff(chroma_norm, axis=1)), axis=0)
        chroma_flux = float(np.mean(frame_flux[valid_pairs]))
    else:
        chroma_flux = 0.0
    return {'chroma_flux': chroma_flux}

def save_highest_similarity_selection(metadata_path, metadata):
    for stale_name in (
        'motif_selections.json',
        'selected_earliest.flac',
        'selected_highest_similarity.flac',
    ):
        stale_path = metadata_path.parent / stale_name
        if stale_path.is_file():
            stale_path.unlink()
    eligible = [
        row for row in metadata['candidates']
        if row['onset_similarity'] >= MIN_ONSET_SIMILARITY
        and row['pitch_class_span'] >= MIN_PITCH_CLASS_SPAN
    ]
    first_by_stem = {}
    for row in sorted(eligible, key=lambda item: item['start_sec']):
        first_by_stem.setdefault(row['stem_name'], row)
    stem_candidates = list(first_by_stem.values())
    if not stem_candidates:
        skipped_metadata = {
            'song_id': metadata['song_id'],
            'title': metadata['title'],
            'source_audio': metadata['source_audio'],
            'status': 'skipped',
            'reason': 'no_candidate_passed_onset_and_pitch_class_span',
            'min_active_ratio': MIN_ACTIVE_RATIO,
            'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
            'onset_threshold': MIN_ONSET_SIMILARITY,
            'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
            'onset_variation_threshold': MIN_ONSET_VARIATION,
            'candidate_count': len(metadata['candidates']),
            'eligible_candidate_count': 0,
            'first_by_stem': {},
            'selections': {},
        }
        (metadata_path.parent / 'motif_selections.json').write_text(
            json.dumps(skipped_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print(f'  [skipped] onset/pitch class span 통과 후보 없음: {metadata["title"]}')
        return skipped_metadata

    for row in stem_candidates:
        candidate_path = metadata_path.parent / row['candidate_file']
        row.update(internal_candidate_features(candidate_path))

    highest_similarity = dict(max(stem_candidates, key=lambda row: row['similarity']))
    final_candidate_path = metadata_path.parent / highest_similarity['candidate_file']
    final_audio, final_sample_rate = librosa.load(
        final_candidate_path, sr=None, mono=False
    )
    final_variation = onset_variation(final_audio, final_sample_rate)
    highest_similarity['onset_variation'] = final_variation
    if final_variation < MIN_ONSET_VARIATION:
        skipped_metadata = {
            'song_id': metadata['song_id'],
            'title': metadata['title'],
            'source_audio': metadata['source_audio'],
            'status': 'skipped',
            'reason': 'final_candidate_onset_variation_below_threshold',
            'min_active_ratio': MIN_ACTIVE_RATIO,
            'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
            'onset_threshold': MIN_ONSET_SIMILARITY,
            'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
            'onset_variation_threshold': MIN_ONSET_VARIATION,
            'candidate_count': len(metadata['candidates']),
            'eligible_candidate_count': len(stem_candidates),
            'first_by_stem': first_by_stem,
            'rejected_final_candidate': highest_similarity,
            'selections': {},
        }
        (metadata_path.parent / 'motif_selections.json').write_text(
            json.dumps(skipped_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print(
            f'  [skipped] final onset variation {final_variation:.3f} '
            f'< {MIN_ONSET_VARIATION:.3f}: {metadata["title"]}'
        )
        return skipped_metadata
    selection_file = 'selected_highest_similarity.flac'
    shutil.copy2(
        metadata_path.parent / highest_similarity['candidate_file'],
        metadata_path.parent / selection_file,
    )
    highest_similarity['selection_file'] = selection_file
    selection_metadata = {
        'song_id': metadata['song_id'],
        'title': metadata['title'],
        'source_audio': metadata['source_audio'],
        'status': 'selected',
        'min_active_ratio': MIN_ACTIVE_RATIO,
        'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
        'onset_threshold': MIN_ONSET_SIMILARITY,
        'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
        'onset_variation_threshold': MIN_ONSET_VARIATION,
        'first_by_stem': first_by_stem,
        'selections': {'highest_similarity': highest_similarity},
    }
    (metadata_path.parent / 'motif_selections.json').write_text(
        json.dumps(selection_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f'  [selection] highest similarity motif 저장: {metadata["title"]}')
    return selection_metadata

melodysim_encoder = None
melodysim_rows = []

def ensure_downbeats(metadata_path, metadata, audio_path):
    downbeats = metadata.get('downbeats')
    if isinstance(downbeats, list) and len(downbeats) > MOTIF_BARS:
        return downbeats
    _, detected_downbeats = beat_tracker(str(audio_path))
    downbeats = np.asarray(detected_downbeats, dtype=np.float64).reshape(-1).tolist()
    metadata['downbeats'] = downbeats
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return downbeats

def run_melodysim_for_song(audio_path, metadata_path, selection_metadata, downbeats):
    global melodysim_encoder
    if selection_metadata.get('status') != 'selected':
        return None
    selected = selection_metadata['selections']['highest_similarity']
    result_path = metadata_path.parent / 'melodysim_scores.json'
    expected_config = {
        'source_audio': str(audio_path),
        'bars': MOTIF_BARS,
        'candidate_start_sec': selected['start_sec'],
        'candidate_end_sec': selected['end_sec'],
        'downbeats': [float(value) for value in downbeats],
        'model_id': MELODYSIM_MODEL_ID,
        'checkpoint_repo': MELODYSIM_CHECKPOINT_REPO,
        'checkpoint_file': MELODYSIM_CHECKPOINT_FILE,
    }
    if not FORCE_REPROCESS and result_path.is_file():
        try:
            cached = json.loads(result_path.read_text(encoding='utf-8'))
        except (OSError, ValueError, TypeError):
            cached = None
        if cached is not None and all(
            cached.get(key) == value for key, value in expected_config.items()
        ):
            song_rows = cached.get('segments', [])
            melodysim_rows.extend(song_rows)
            print(
                f'  [MelodySim: cached] mean={cached.get("mean_similarity")}: '
                f'{selection_metadata["title"]}'
            )
            display(pd.DataFrame(song_rows))
            return cached

    melodic_audio, melodic_sample_rate = librosa.load(
        metadata_path.parent / 'melodic_accompaniment.flac',
        sr=None,
        mono=False,
    )
    duration_sec = melodic_audio.shape[-1] / melodic_sample_rate
    segment_rows = phase_aligned_four_bar_segments(
        downbeats,
        selected['start_sec'],
        bars=MOTIF_BARS,
        audio_duration_sec=duration_sec,
    )
    if not segment_rows:
        raise ValueError('MelodySim에 사용할 완전한 4마디 구간이 없습니다.')
    candidate_start = round(selected['start_sec'] * melodic_sample_rate)
    candidate_end = round(selected['end_sec'] * melodic_sample_rate)
    candidate_audio = melodic_audio[..., candidate_start:candidate_end]
    segment_audios = [
        melodic_audio[
            ...,
            round(row['start_sec'] * melodic_sample_rate):
            round(row['end_sec'] * melodic_sample_rate),
        ]
        for row in segment_rows
    ]
    if melodysim_encoder is None:
        print('  [MelodySim] MERT-95M과 공개 checkpoint 로드')
        melodysim_encoder = MelodySimEncoder(device=DEVICE)
    similarities = melodysim_encoder.compare_candidate_to_segments(
        candidate_audio,
        segment_audios,
        sample_rate=melodic_sample_rate,
        batch_size=MELODYSIM_BATCH_SIZE,
    )
    song_rows = []
    for row, similarity in zip(segment_rows, similarities):
        song_rows.append({
            'song_id': selection_metadata['song_id'],
            'title': selection_metadata['title'],
            **row,
            'melodysim_similarity': similarity,
        })
    non_source_scores = [
        row['melodysim_similarity'] for row in song_rows if not row['is_source']
    ]
    mean_similarity = (
        float(np.mean(non_source_scores)) if non_source_scores else None
    )
    for row in song_rows:
        row['mean_similarity_excluding_source'] = mean_similarity
    payload = {
        **expected_config,
        'title': selection_metadata['title'],
        'source_segment_excluded_from_mean': True,
        'mean_similarity': mean_similarity,
        'segments': song_rows,
    }
    result_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    melodysim_rows.extend(song_rows)
    print(
        f'  [MelodySim] mean(excluding source)={mean_similarity}: '
        f'{selection_metadata["title"]}'
    )
    display(pd.DataFrame(song_rows))
    return payload

all_rows = []
dataset_results = []
pending_jobs = []
assert DEMUCS_BATCH_SIZE >= 1
for song_index, audio_path in enumerate(song_paths):
    song = songs[song_index]
    display_title = f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    print(f'[{song_index + 1}/{len(song_paths)}] {display_title}')
    song_id = f'{song_index:03d}_{safe_folder_name(display_title)}'
    song_dir = DIAGNOSTIC_DIR / song_id
    existing_song_dir = find_existing_song_dir(audio_path)
    if existing_song_dir is not None and existing_song_dir != song_dir:
        if not song_dir.exists():
            existing_song_dir.rename(song_dir)
            print(f'  [rename] {existing_song_dir.name} -> {song_dir.name}')
        else:
            song_dir = existing_song_dir
            song_id = song_dir.name
            print(f'  [warning] 대상 폴더가 이미 있어 기존 폴더 유지: {song_dir.name}')
    song_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = song_dir / 'motif_scores.json'

    existing = None
    if not FORCE_REPROCESS and metadata_path.is_file() and (song_dir / 'melodic_accompaniment.flac').is_file():
        try:
            candidate_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            same_audio = (
                Path(candidate_metadata.get('source_audio', '')).resolve()
                == Path(audio_path).resolve()
            )
            candidates = candidate_metadata.get('candidates')
            candidates_complete = isinstance(candidates, list) and all(
                'pitch_class_span' in row
                and row.get('candidate_file')
                and (song_dir / row['candidate_file']).is_file()
                for row in candidate_metadata.get('candidates', [])
            )
            if same_audio and candidates_complete:
                existing = candidate_metadata
        except (OSError, ValueError, TypeError):
            existing = None

    if existing is not None:
        selection_path = song_dir / 'motif_selections.json'
        try:
            cached_selection = json.loads(selection_path.read_text(encoding='utf-8'))
        except (OSError, ValueError, TypeError):
            cached_selection = None
        if cached_selection is not None and cached_selection.get('status') == 'skipped':
            same_skip_audio = (
                Path(cached_selection.get('source_audio', '')).resolve()
                == Path(audio_path).resolve()
            )
            same_skip_config = all((
                cached_selection.get('min_active_ratio') == MIN_ACTIVE_RATIO,
                cached_selection.get('max_onset_chroma_difference') == MAX_ONSET_CHROMA_DIFFERENCE,
                cached_selection.get('onset_threshold') == MIN_ONSET_SIMILARITY,
                cached_selection.get('pitch_class_span_threshold') == MIN_PITCH_CLASS_SPAN,
                cached_selection.get('onset_variation_threshold') == MIN_ONSET_VARIATION,
            ))
            if same_skip_audio and same_skip_config:
                for row in existing['candidates']:
                    all_rows.append({'song_id': song_id, 'title': display_title, **row})
                print(f'  [skip: cached] 이전 실행에서 통과한 모티프 없음: {display_title}')
                continue
            print('  [reprocess] 스킵 이후 음원 또는 임계값 변경')
            existing = None

    if existing is not None:
        if dataset_builder is not None:
            dataset_results.append(dataset_builder.process_track(song['track'], audio_index))
        elif not song_stems_complete(song, audio_path):
            print('  [process] 저장되지 않은 songs stem 분리')
            stems, sample_rate, mixture = separator.separate(audio_path)
            save_song_stems(song, audio_path, stems, sample_rate)
            del stems, mixture
        print('  [reuse] 기존 분리/후보 결과 사용')
        for row in existing['candidates']:
            all_rows.append({'song_id': song_id, 'title': display_title, **row})
        downbeats = ensure_downbeats(metadata_path, existing, audio_path)
        selection_metadata = save_highest_similarity_selection(metadata_path, existing)
        run_melodysim_for_song(
            audio_path, metadata_path, selection_metadata, downbeats
        )
        continue

    pending_jobs.append({
        'song_index': song_index, 'song': song, 'audio_path': audio_path,
        'display_title': display_title, 'song_id': song_id,
        'song_dir': song_dir, 'metadata_path': metadata_path,
    })

for batch_start in range(0, len(pending_jobs), DEMUCS_BATCH_SIZE):
    batch_jobs = pending_jobs[batch_start:batch_start + DEMUCS_BATCH_SIZE]
    batch_paths = [job['audio_path'] for job in batch_jobs]
    batch_number = batch_start // DEMUCS_BATCH_SIZE + 1
    batch_count = (len(pending_jobs) + DEMUCS_BATCH_SIZE - 1) // DEMUCS_BATCH_SIZE
    print(f'[Demucs batch {batch_number}/{batch_count}] {len(batch_jobs)}곡 분리')
    batch_separations = separator.separate_many(batch_paths)
    assert len(batch_separations) == len(batch_jobs)

    for job, separation_result in zip(batch_jobs, batch_separations):
        song = job['song']
        audio_path = job['audio_path']
        display_title = job['display_title']
        song_id = job['song_id']
        song_dir = job['song_dir']
        metadata_path = job['metadata_path']
        stems, sample_rate, mixture = separation_result
        print(f'  [motif] {display_title}')
        result = motif_scorer.score_all(audio_path, stems, sample_rate)
        if dataset_builder is not None:
            dataset_results.append(dataset_builder.process_track(
                song['track'],
                audio_index,
                separation_result=separation_result,
                motif_scores=result,
            ))
        else:
            save_song_stems(song, audio_path, stems, sample_rate)
        melodic = result['melodic_accompaniment']
        save_audio(song_dir / 'melodic_accompaniment.flac', melodic, sample_rate, audio_format='flac')

        for row_index, row in enumerate(result['candidates']):
            start = round(row['start_sec'] * sample_rate)
            end = round(row['end_sec'] * sample_rate)
            filename = f"candidate_{row_index:03d}_{row['stem_name']}.flac"
            save_audio(song_dir / filename, melodic[:, start:end], sample_rate, audio_format='flac')
            row['candidate_file'] = filename
            all_rows.append({'song_id': song_id, 'title': display_title, **row})

        metadata = {
            'song_id': song_id, 'title': display_title, 'source_audio': str(audio_path),
            'sample_rate': sample_rate, 'candidates': result['candidates'],
            'downbeats': result['downbeats'],
        }
        metadata_path.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        selection_metadata = save_highest_similarity_selection(metadata_path, metadata)
        run_melodysim_for_song(
            audio_path, metadata_path, selection_metadata, result['downbeats']
        )
        del stems, mixture, result, melodic

scores_df = pd.DataFrame(all_rows)
scores_df.to_csv(DIAGNOSTIC_DIR / 'all_motif_scores.csv', index=False)
display(scores_df.sort_values(['song_id', 'start_sec', 'stem_name']))
if dataset_results:
    from dataclasses import asdict
    display(pd.DataFrame([asdict(result) for result in dataset_results]))
melodysim_df = pd.DataFrame(melodysim_rows)
melodysim_df.to_csv(DIAGNOSTIC_DIR / 'melodysim_scores.csv', index=False)
if not melodysim_df.empty:
    display(melodysim_df.sort_values(['song_id', 'segment_index']))

## 4. 현재 곡의 highest similarity motif 일괄 재생

현재 입력에 포함된 곡 중 저장된 highest similarity motif를 `DEBUG_TRACK_INDEX`부터 `DEBUG_TRACK_COUNT`곡씩 표시합니다. 예: 11~20곡은 `DEBUG_TRACK_INDEX = 10`으로 설정합니다.

In [ ]:
from IPython.display import Audio, display

DEBUG_TRACK_INDEX = 0
DEBUG_TRACK_COUNT = 10
selection_paths = []
for audio_path in song_paths:
    selected_song_dir = find_existing_song_dir(audio_path)
    if selected_song_dir is None:
        continue
    selection_path = selected_song_dir / 'motif_selections.json'
    if selection_path.is_file():
        selection_metadata = json.loads(selection_path.read_text(encoding='utf-8'))
        if selection_metadata.get('status') == 'skipped':
            print(f'[skipped] {selection_metadata["title"]}: {selection_metadata["reason"]}')
            continue
        selection_paths.append(selection_path)
assert selection_paths, '현재 곡에 재생 가능한 motif selection이 없습니다.'
selected_paths = selection_paths[
    DEBUG_TRACK_INDEX:DEBUG_TRACK_INDEX + DEBUG_TRACK_COUNT
]
assert selected_paths, (
    f'재생할 곡이 없습니다: start={DEBUG_TRACK_INDEX}, total={len(selection_paths)}'
)

for track_number, selection_path in enumerate(
    selected_paths, start=DEBUG_TRACK_INDEX + 1
):
    selection_metadata = json.loads(selection_path.read_text(encoding='utf-8'))
    row = selection_metadata['selections']['highest_similarity']
    print(f'\n{track_number}. {selection_metadata["title"]}')
    print({key: value for key, value in row.items() if key not in {'candidate_file', 'selection_file'}})
    display(Audio(filename=str(selection_path.parent / row['selection_file'])))